# Lab 5 — Example: Preprocessing & Feature Engineering

**ITCS 3162 — Introduction to Data Mining**
**Companion to Zybooks Module 3 (Preprocessing)**

In Labs 3 and 4 you wrangled and explored data. **Preprocessing** is the next bridge — turning a tidy DataFrame into something a machine learning model can actually consume. The themes:

- Drop columns and rows that shouldn't be there
- Handle missing values (with eyes open about the trade-offs)
- Encode categorical variables — and pick the right kind of encoding
- Scale numeric variables — and avoid leakage between train and test
- Recognize correlated and low-importance features
- Bundle the whole thing as a reusable `Pipeline`

## Learning objectives
After this lab you will be able to:
1. Identify and remove ID-like columns, near-constants, and target-leakers
2. Choose between dropping and imputing missing values, and justify the choice
3. Apply **ordinal** vs. **one-hot** encoding based on whether a category has natural order
4. Use `StandardScaler` and `MinMaxScaler` correctly (without data leakage)
5. Detect highly correlated features and decide what to do with them
6. Use a tree model's `feature_importances_` to spot low-value features
7. Compose all of the above into a single `sklearn.Pipeline`

We'll use the **Titanic dataset** — it's small but rich: it has missing values in different patterns, a mix of nominal and ordinal categoricals, an obvious ID column, and a target-leaker hiding in plain sight.


## 1. Load and survey


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

df = sns.load_dataset("titanic")
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

**Goal:** predict `survived` (0 = died, 1 = survived). Some columns are obvious features (age, fare, sex); others need scrutiny.


## 2. Drop columns that don't belong

Before any modeling, scan the columns for three kinds of trouble:

| Trouble | Example here | Why it's a problem |
|---|---|---|
| **Target leakers** — encode the label | `alive` ("yes"/"no") | The model would cheat by reading this directly |
| **Redundant duplicates** — same info as another column | `class` duplicates `pclass`, `embark_town` duplicates `embarked`, `who`/`adult_male` derived from `age` + `sex` | Don't add information; can confuse interpretation |
| **ID-like columns** — unique per row | Titanic doesn't have one, but **passenger IDs** in the original dataset are this kind | Memorizing IDs doesn't generalize |

Drop them now. Better to be aggressive — you can add back later if you discover you need something.


In [ ]:
# Confirm 'alive' is a perfect leak before dropping
print("alive vs. survived:")
print(pd.crosstab(df["alive"], df["survived"]))

`alive == "yes"` iff `survived == 1` — perfectly redundant with the target.


In [ ]:
to_drop = [
    "alive",         # target leaker
    "class",         # redundant with pclass
    "embark_town",   # redundant with embarked
    "who",           # derived from age + sex
    "adult_male",    # derived from age + sex
    "alone",         # derived from sibsp + parch
]
df = df.drop(columns=to_drop)
print("After drop:", df.shape)
print("Remaining columns:", list(df.columns))

## 3. Handle missing values

Always start with the count:


In [ ]:
miss = pd.DataFrame({
    "n_missing": df.isna().sum(),
    "pct_missing": (df.isna().sum() / len(df) * 100).round(1),
}).sort_values("n_missing", ascending=False)
miss

Three columns have missing data, in very different patterns:

| Column | Missing | Decision |
|---|---|---|
| `deck` | 688 / 891 (77%) | **Drop the column** — too sparse to be useful |
| `age` | 177 / 891 (20%) | **Impute** — too valuable to throw away, but we'll be careful |
| `embarked` | 2 (0.2%) | **Drop those rows** — losing 2 of 891 is fine |

### Decision rules I use as defaults

1. **>50% missing** → drop the column. You can't reliably fill that much.
2. **Whole rows missing the target** → drop them. Imputing the *target* corrupts your training data.
3. **A handful (<5%) missing in any single column** → drop those rows.
4. **A moderate amount missing in a feature column** → impute, but understand it adds bias.

> ⚠️ Imputation is not free. Filling in `age` with the median acts like you're confident those passengers were 28 years old. The model will trust the imputed values as much as the real ones. For high-stakes use cases, add a binary `age_missing` indicator so the model knows which rows were imputed.


In [ ]:
# Step 1: drop the sparse deck column
df = df.drop(columns=["deck"])

# Step 2: drop the two rows with missing embarked
df = df.dropna(subset=["embarked"])

# Step 3: impute age. Add an indicator first so the model knows.
df["age_missing"] = df["age"].isna().astype(int)
df["age"] = df["age"].fillna(df["age"].median())

print("After missing-value handling:", df.shape)
print("Missing remaining:", df.isna().sum().sum())

### Quick aside on `mean` vs. `median` vs. `mode`

- **Mean** for symmetric numeric data without outliers.
- **Median** for skewed numeric data (e.g. income, fare). More robust to outliers. Most defaults should be median.
- **Mode** (most common value) for categorical data. `df["embarked"].fillna(df["embarked"].mode()[0])`.

There are smarter imputers (`KNNImputer`, `IterativeImputer`) — sklearn provides them — but for an intro lab, median/mode covers the common case.


## 4. Encode categorical variables

Models need numbers. Two main strategies, and **picking the right one matters**:

### Ordinal encoding — when there's a natural order

`pclass`: First (1) > Second (2) > Third (3) is an order. The numeric distance is meaningful (sort of — first to second isn't necessarily the same gap as second to third, but the order is real).

### One-hot encoding — when there's no order

`embarked`: C, Q, S are three ports. No order. If you encode them as 0, 1, 2, you'd be telling the model that Q is "between" C and S, which is meaningless.


In [ ]:
# pclass is already 1/2/3 in this dataset — great, that IS ordinal encoding done for us.
print("pclass unique values:", df["pclass"].unique())
print("dtype:", df["pclass"].dtype)

In [ ]:
# sex is binary nominal — we can encode 0/1, or one-hot. For binary it's equivalent.
df["sex"] = (df["sex"] == "female").astype(int)
df["sex"].head()

In [ ]:
# embarked has 3 unordered levels — one-hot encode it.
df = pd.get_dummies(df, columns=["embarked"], prefix="embarked", drop_first=False)
df.head()

About `drop_first`: setting it to `True` would drop one level (e.g. `embarked_C`) to avoid perfect multicollinearity. Required for linear models with an intercept; safe to keep all levels for tree models. We'll leave them in here.

### What about `LabelEncoder`?

You'll see `LabelEncoder` in tutorials. **`LabelEncoder` is for the *target*, not for features.** Using it on features is a common bug — it secretly applies ordinal encoding to nominal data. Use `OrdinalEncoder` (with an explicit category order) for ordinal features, and `OneHotEncoder` (or `pd.get_dummies`) for nominal.


## 5. Scale numeric features

After encoding, our numeric features are on wildly different scales:

- `age`: ~0–80
- `fare`: ~0–512
- `pclass`: 1–3
- `sibsp`, `parch`: 0–8

Models that compute distances (k-NN) or use gradient descent (logistic regression, neural nets) are sensitive to this. **Tree-based models (decision trees, random forests) are not** — they look at thresholds per feature, not magnitudes.

Two common scalers:

| Scaler | Output | Use when |
|---|---|---|
| `StandardScaler` | mean=0, std=1 | Features are roughly normal-ish; default for most models |
| `MinMaxScaler` | values in [0, 1] | You need bounded values (e.g., neural nets); when the original scale matters |

### The leakage trap

A common mistake: scaling the entire dataset *before* splitting into train and test. That lets test-set statistics leak into training. The right order:

1. `train_test_split` first
2. `fit` the scaler on training data only
3. `transform` both training and test data with that fitted scaler


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=["survived"])
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Fit on TRAIN ONLY, then transform both
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

num_cols = ["age", "fare", "sibsp", "parch", "pclass"]
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols]  = scaler.transform(X_test[num_cols])

X_train_scaled[num_cols].describe().round(2)

Numeric features now have mean ≈ 0, std ≈ 1 (on training data). Test features were transformed using the *training* statistics, which is correct.


## 6. Highly correlated features


In [ ]:
# Correlation matrix on training features
corr = X_train_scaled.corr().round(2)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f", ax=ax)
ax.set_title("Feature correlations (training set)")
plt.show()

Nothing screams here. `sibsp` and `parch` correlate weakly (~0.4) — both relate to family size. Some `embarked_*` dummies are negatively correlated with each other by construction (one being 1 forces others to 0).

### What to do when you DO find correlated pairs (e.g. > 0.85)

- **Drop one of the pair** if they truly carry the same information.
- **Combine them** into a new feature (e.g. `family_size = sibsp + parch`).
- **Leave them** if you're using a tree model (it'll just pick one).
- **Use PCA** to combine many correlated features into fewer components — more on that below.


In [ ]:
# A useful feature-engineering example: combine sibsp + parch
X_train_scaled["family_size"] = X_train_scaled["sibsp"] + X_train_scaled["parch"]
X_test_scaled["family_size"]  = X_test_scaled["sibsp"]  + X_test_scaled["parch"]
X_train_scaled[["sibsp","parch","family_size"]].head()

## 7. Feature importance — quick look ahead

We'll dive into Random Forest in Lab 7. For now, here's the punchline: **tree-based models tell you how useful each feature was**. That's a fast way to spot features that aren't pulling their weight.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Cast bool dummies to int for the model
X_tr = X_train_scaled.astype({c: int for c in X_train_scaled.select_dtypes("bool").columns})

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_tr, y_train)

importances = pd.Series(rf.feature_importances_, index=X_tr.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot.barh(ax=ax)
ax.set_title("Random Forest feature importance")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

`sex` and `fare` dominate. `parch`, `age_missing`, and the embarked dummies barely register. In practice, you might consider dropping the bottom-importance features for a simpler, more interpretable model.

> **Trees handle redundant and irrelevant features automatically** — they just don't split on them. That's a real strength versus linear models, where uninformative features still add coefficients to fit. We'll see this pay off in Lab 7.


## 8. A note on PCA (dimensionality reduction)

When you have **many** numeric features that are correlated with each other, you can compress them with **Principal Component Analysis (PCA)** — a math trick that finds new axes that capture the most variance.

For this lab the dataset is small enough that PCA isn't worth it. But the one-line version:

```python
from sklearn.decomposition import PCA
pca = PCA(n_components=3)
X_reduced = pca.fit_transform(X_train_scaled[num_cols])
print(pca.explained_variance_ratio_)   # how much variance each component captures
```

PCA components are linear combinations of your original features — interpretable for low-D data, harder to make sense of in high-D. Use it when:
- You have many highly correlated numeric features
- You want to visualize high-dim data in 2D (`n_components=2`)
- You're feeding a model that struggles with high dimensionality

It's not a magic fix. Trees and modern gradient-boosting models don't usually need it. We'll leave deeper PCA for later.


## 9. Putting it together — sklearn `Pipeline` + `ColumnTransformer`

So far we did everything step-by-step. In practice you want this packaged so you can apply the *same* preprocessing to training data, test data, and future production data without bugs. sklearn's `Pipeline` and `ColumnTransformer` are the tools.

`ColumnTransformer` applies different preprocessing to different columns. `Pipeline` chains preprocessing → model.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# Start fresh — reload, drop the obvious garbage, and split immediately.
df_raw = sns.load_dataset("titanic")
df_raw = df_raw.drop(columns=["alive","class","who","adult_male","embark_town","alone","deck"])
df_raw = df_raw.dropna(subset=["embarked"])  # only 2 rows; cheaper than imputing

X = df_raw.drop(columns=["survived"])
y = df_raw["survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Identify column groups
num_cols = ["age", "fare"]
nominal_cols = ["sex", "embarked"]
ordinal_cols = ["pclass", "sibsp", "parch"]   # already numeric-coded

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale",  StandardScaler()),
])

nominal_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num",     numeric_pipe, num_cols),
    ("nom",     nominal_pipe, nominal_cols),
    ("ordinal", "passthrough", ordinal_cols),
])

full_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model",      LogisticRegression(max_iter=500)),
])

full_pipeline.fit(X_train, y_train)
acc = full_pipeline.score(X_test, y_test)
print(f"Test accuracy: {acc:.3f}")

Look at what this pipeline does in one line of `fit`:

1. **Impute** missing `age`/`fare` with the median (fit on training data only — automatic)
2. **Scale** numeric columns to zero mean and unit variance (fit on training only — automatic)
3. **Impute** missing `embarked` with the most common value (fit on training only — automatic)
4. **One-hot encode** the nominal columns (fit on training only — automatic)
5. **Pass through** the ordinal columns unchanged
6. **Fit** a logistic regression on the combined feature matrix
7. **Predict** on the test set with all the same transformations applied

No leakage. No "did I scale the test set with the wrong statistics?" Every preprocessing step is fit on training only and reapplied identically to test data. **This is the right way to do preprocessing.**


## Summary

You walked through the full preprocessing playbook:

| Step | Tool |
|---|---|
| Drop ID / leaker / duplicate columns | `df.drop(columns=[...])` |
| Drop sparse columns (>50% missing) | `df.drop` |
| Drop tiny # of missing rows | `df.dropna(subset=...)` |
| Impute the rest | `SimpleImputer` with median/mode |
| Encode ordinal | already-numeric, or `OrdinalEncoder` with explicit order |
| Encode nominal | `pd.get_dummies` or `OneHotEncoder` |
| Scale numeric | `StandardScaler` (default) or `MinMaxScaler` |
| Spot correlated features | `df.corr()` + heatmap |
| Identify weak features | `RandomForestClassifier().feature_importances_` |
| Compose end-to-end | `Pipeline` + `ColumnTransformer` |

Now open **`lab_05_exercise.ipynb`** and apply this playbook to the Adult Census Income dataset.
